In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import json
import re
from pathlib import Path
from typing import Optional, List

import pandas as pd

# ========= CONFIG =========
INPUT_CSV = "survey_data.csv"
OUT_PROFILES = "survey_profiles.json"
OUT_STYLE = "survey_student_style.txt"

DROP_COL_CONTAINS = ["random id"]
IGNORE_COLS_EXACT = {"Q33_5_TEXT"}

ALIASES = {
    "course_load_term": [
        "How heavy is your courseload this academic term?"
    ],
    "course_load_typical": [
        "How heavy is your typical courseload?"
    ],
    "gpa": [
        "What is your current GPA?"
    ],
    "tutoring_now": [
        "Currently, how often do you use school-provided tutoring?"
    ],
    "advising_now": [
        "Currently, not counting mandatory advising meetings, how often do you use academic advising?"
    ],
    "sleep_hours": [
        "About how many hours per week do you spend on each of these activities? - Weekly Sleeping"
    ],
    "work_hours": [
        "About how many hours per week do you spend on each of these activities? - Other Work Obligations (employment, volunteer work, entrepreneurship, etc.)"
    ],
    "continuation_intent": [
        "How would you rate your intention to continue your program next academic term?"
    ],
    "confidence": [
        "How would you rate your confidence that you will complete your computing degree?"
    ],
    # Open-text style fields
    "conflict_which": [
        "Which activities, if any, most often conflict with your schoolwork?"
    ],
    "conflict_how": [
        "How do you usually resolve any such conflicts?"
    ],
    "leave_reason_other_text": [
        "If you were to leave your computing program in the future, what would be the most likely reason for your decision? - Other (please specify) - Text"
    ],
    "leave_instead": [
        "If you left your computing program, what would you do instead?"
    ],
}

COURSE_LOAD_MAP = {
    "Less than half couseload": 1,
    "Less than full but at least half courseload": 2,
    "Full courseload": 3,
}
CGPA_MAP = {
    "0 - 2.19": 1,
    "2.2 - 2.79": 2,
    "2.8 - 3.29": 3,
    "3.3 - 4.0": 4,
}
FREQ_MAP = {
    "Never": 0,
    "At least once in the semester": 1,
    "At least once a month": 2,
    "At least once a week": 3,
    "Daily": 4,
}
INTENT_MAP = {
    "I fully intend to": 3,
    "I mostly intend to": 2,
    "I am unsure": 1,
    "I do not intend to": 0,
}
CONF_MAP = {
    "Very confident": 3,
    "Mostly confident": 2,
    "Somewhat confident": 1,
    "Not at all confident": 0,
}

def derive_strength(cgpa, confidence, course_load_term):
    cg = cgpa if isinstance(cgpa, (int, float)) and cgpa is not None else 0
    conf = confidence if isinstance(confidence, (int, float)) and confidence is not None else 0
    cl = course_load_term if isinstance(course_load_term, (int, float)) and course_load_term is not None else 0
    if cg <= 2 or conf <= 1 or cl == 1:
        return "weak"
    if cg >= 4 and conf >= 2:
        return "strong"
    return "average"

# ========= HELPERS =========
def pick_col(df: pd.DataFrame, options: List[str]) -> Optional[str]:
    """Find a column by exact match, then forgiving prefix (case-insensitive)."""
    if not options:
        return None
    for o in options:
        if o in df.columns:
            return o
    lowers = {c.lower(): c for c in df.columns}
    for o in options:
        pref = o.strip().lower()[:50]
        for cl in lowers:
            if cl.startswith(pref):
                return lowers[cl]
    return None

def map_exact(val, mapping, default=None):
    if pd.isna(val):
        return default
    s = str(val).strip()
    return mapping.get(s, default)

def to_number(val, default=None):
    if pd.isna(val):
        return default
    nums = re.findall(r"-?\d+\.?\d*", str(val))
    if not nums:
        return default
    try:
        v = float(nums[0])
        return int(v) if v.is_integer() else v
    except Exception:
        return default

# ========= MAIN =========
def main():
    in_path = Path(INPUT_CSV)
    if not in_path.exists():
        raise FileNotFoundError(f"CSV not found at: {in_path.resolve()}")

    df = pd.read_csv(in_path, header=1, encoding="utf-8", low_memory=False)

    # Drop random-id and explicitly ignored columns
    drop_cols = []
    for c in df.columns:
        lc = str(c).strip().lower()
        if any(k in lc for k in DROP_COL_CONTAINS):
            drop_cols.append(c)
        if c in IGNORE_COLS_EXACT:
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=list(set(drop_cols)), errors="ignore")

    cols = {key: pick_col(df, opts) for key, opts in ALIASES.items()}

    profiles = []
    for i, row in df.iterrows():
        cgpa_num = map_exact(row.get(cols["gpa"]), CGPA_MAP, default=None) if cols["gpa"] else None
        cl_term = map_exact(row.get(cols["course_load_term"]), COURSE_LOAD_MAP, default=None) if cols["course_load_term"] else None
        tutoring_now = map_exact(row.get(cols["tutoring_now"]), FREQ_MAP, default=None) if cols["tutoring_now"] else None
        advising_now = map_exact(row.get(cols["advising_now"]), FREQ_MAP, default=None) if cols["advising_now"] else None
        sleep_hours = to_number(row.get(cols["sleep_hours"]), default=None) if cols["sleep_hours"] else None
        work_hours = to_number(row.get(cols["work_hours"]), default=None) if cols["work_hours"] else None
        intention = map_exact(row.get(cols["continuation_intent"]), INTENT_MAP, default=None) if cols["continuation_intent"] else None
        confidence = map_exact(row.get(cols["confidence"]), CONF_MAP, default=None) if cols["confidence"] else None

        strength = derive_strength(cgpa_num, confidence, cl_term)

        rec = {
            "student_id": i + 1,
            "cgpa": cgpa_num,
            "course_load": cl_term,
            "confidence": confidence,
            "continuation_intent": intention,
            "tutoring_use": tutoring_now,
            "advising_use": advising_now,
            "sleep_cutback": sleep_hours,
            "work_hours": work_hours,
            "profile_strength": strength,
        }
        profiles.append(rec)

    with open(OUT_PROFILES, "w", encoding="utf-8") as f:
        json.dump(profiles, f, indent=2, ensure_ascii=False)

    blocks = []
    for i, row in df.iterrows():
        parts = []
        for key in ("conflict_which", "conflict_how", "leave_instead", "leave_reason_other_text"):
            col = cols.get(key)
            if not col:
                continue
            val = str(row.get(col, "")).strip()
            if not val:
                continue
            label = {
                "conflict_which": "Conflicts",
                "conflict_how": "Resolve",
                "leave_instead": "If left",
                "leave_reason_other_text": "Leave-Reason-Other",
            }[key]
            parts.append(f"{label}: {val}")
        block = "\n".join(parts).strip()
        if block:
            blocks.append(f"[Student {i+1}]\n{block}\n")

    Path(OUT_STYLE).write_text("\n".join(blocks), encoding="utf-8")

    print(f"Wrote {len(profiles)} profiles → {OUT_PROFILES}")
    print(f"Wrote style file with {len(blocks)} blocks → {OUT_STYLE}")

if __name__ == "__main__":
    main()


Wrote 156 profiles → survey_profiles.json
Wrote style file with 156 blocks → survey_student_style.txt


In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import json
from pathlib import Path
import pandas as pd

IN_JSON = "survey_profiles.json"

def main():
    p = Path(IN_JSON)
    if not p.exists():
        raise FileNotFoundError(f"Couldn't find {IN_JSON}. Run the profiling script first.")

    with p.open("r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data)
    counts = (
        df["profile_strength"]
        .value_counts(dropna=False)
        .rename_axis("strength")
        .reset_index(name="count")
    )
    counts["percent"] = (counts["count"] / counts["count"].sum() * 100).round(2)

    # Print nicely
    print("Profile Strength Counts:")
    for _, r in counts.iterrows():
        print(f"- {r['strength'].capitalize()}: {r['count']} ({r['percent']}%)")

    # Print a compact table
    print("\nTable:")
    print(counts.to_string(index=False))

if __name__ == "__main__":
    main()


Profile Strength Counts:
- Weak: 81 (51.92%)
- Strong: 59 (37.82%)
- Average: 16 (10.26%)

Table:
strength  count  percent
    weak     81    51.92
  strong     59    37.82
 average     16    10.26
